# Imports

In [1]:
# loading libraries for data manipulation
import numpy as np
import pandas as pd

# loading libraries for data visualization
import matplotlib.pyplot as plt
from plotnine import *
from PIL import Image

# import tensorflow and keras packages
import tensorflow as tf
from tensorflow import keras

# let's also include different Models, Layers directly from keras
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.layers import Dense,Dropout,LSTM,Embedding,Input,GRU

# use requests package to download some text
import requests

import warnings
warnings.filterwarnings('ignore')

# Data Retrieval/Cleaning

In [2]:
url = "https://gutenberg.org/cache/epub/1513/pg1513.txt" # Romeo and Juliet
text = requests.get(url).text

# clean text 
text = text[text.find("THE PROLOGUE")+10:text.find("*** END OF THE PROJECT")] # exclude metadata
text = text.lower()
print(f"Length of text: {len(text)} characters")

Length of text: 147650 characters


In [3]:
words = text.split()
print(f"Total words: {len(words)}")

Total words: 25947


In [4]:
vocab = sorted(set(words))
print(f"Unique words: {len(vocab)}")

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}

Unique words: 5726


In [5]:
text_as_int = np.array([word2idx[w] for w in words], dtype=np.int32)
print("First 20 encoded words:", text_as_int[:20])

First 20 encoded words: [5130   89 2413 4132 2416   46 3828 3666 4132 2426   46 4646 4132 2428
 4034 2447  762 2376 4132 2512]


In [17]:
seq_length = 15  # smaller since words carry more info
examples_per_epoch = len(text_as_int) // (seq_length + 1)
print(f"Number of sequences: {examples_per_epoch}")

Number of sequences: 1621


In [18]:
word_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = word_dataset.batch(seq_length + 1, drop_remainder=True)

In [19]:
def split_input_target(chunk):
    input_seq = chunk[:-1]
    target_seq = chunk[1:]
    return input_seq, target_seq

# apply the function to sequences
dataset = sequences.map(split_input_target)

In [20]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

# Initial Network

In [21]:
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 512

model = Sequential([
    Input(shape=(None,)),
    Embedding(vocab_size, embedding_dim),
    LSTM(rnn_units, return_sequences=True),
    Dropout(0.4),
    Dense(vocab_size)
])

model.compile(
    optimizer='adam',
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True)
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, None, 256)      │     1,465,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, None, 512)      │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, None, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, None, 5726)     │     2,937,438 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,978,206 (22.81 MB)

 Trainable params: 5,978,206 (22.81 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history = model.fit(dataset, epochs=20,verbose=1)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 370ms/step - loss: 5.6446
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 8s 320ms/step - loss: 5.5569
Epoch 3/20
12/25 ━━━━━━━━━━━━━━━━━━━━ 4s 347ms/step - loss: 5.4024

KeyboardInterrupt: 

# Generating Text From Prelim Model

In [24]:
def generate_text(model, start_seq, num_generate=50, temperature=1.0):
    # Tokenize the starting sequence into words
    input_eval = [word2idx.get(w, 0) for w in start_seq.lower().split()]
    input_eval = tf.expand_dims(input_eval, 0)

    generated_words = []

    for _ in range(num_generate):
        predictions = model.predict(input_eval, verbose=0)
        predictions = tf.squeeze(predictions, 0)
        predictions = predictions / temperature

        predicted_id = tf.random.categorical(predictions[-1:], num_samples=1)[0, 0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)
        generated_words.append(idx2word[predicted_id])

    return start_seq + ' ' + ' '.join(generated_words)

In [25]:
generate_text(model, "romeo felt", 500, 1.0)

'romeo felt for envious? partizans._] come descry. money, simple hair chambermaids. wise, night; merchandise. hearts, heel wake, benvolio. for settled gentler kiss wounded rocks church, nose passado, fright notice plate. [_enters the affords you, underneath a cave? pump, thou strew take a anger bring likewise cats. thin whate’er married doth on my maidenheads; ready? her am: thy hand wife. festival affections’ ensign montague, the word. potpan! sweet. faith, come case. fetch early? traces, part!’ i i pale lineament, partisans, leg cannot. nine beat delay to arm heap’d it then this riband? madness sighs, text. window, i am green youth again; can help! we black mend. action here or our hoodwink’d crutch! points, drawn, it,—here should hear put pathway, what hast prevails then. proof alla learned. in out was thou, make body’s fly. exhales gentle page? perchance mercutio’s which that it will one unattainted her live paradise, younger darkness with have devise like, afraid paris your she,— 